In [53]:
import pandas as pd
from sklearn.metrics import classification_report
import ast

In [13]:
df_1 = pd.read_csv('evaluated_shape_mistral_template_1.csv',)
df_2 = pd.read_csv('evaluated_shape_mistral_template_2.csv')
df_3 = pd.read_csv('evaluated_shape_mistral_template_3.csv')
df_4 = pd.read_csv('evaluated_shape_mistral_template_4.csv')
test = pd.read_csv('../test/shape.csv')

In [11]:
def extract_first_digit(s):
    digit = ''.join([c for c in s if c.isdigit()])[0]
    return digit

In [ ]:
def clean_predictions(prediction):
    prediction = prediction.apply(lambda x: extract_first_digit(x))
    return prediction
# df_1['predictions_1'] = df_1['predictions_1'].apply(lambda x: extract_first_digit(x))
# df_2['predictions_2'] = df_2['predictions_2'].apply(lambda x: extract_first_digit(x))
# df_3['predictions_3'] = df_3['predictions_3'].apply(lambda x: extract_first_digit(x))
# df_4['predictions_4'] = df_4['predictions_4'].apply(lambda x: extract_first_digit(x))

IndexError: string index out of range

In [73]:
full_df = pd.DataFrame({
    'predictions_1': df_1['predictions_1'],
    'predictions_2': df_2['predictions_2'],
    'predictions_3': df_3['predictions_3'],
    'label': test['label']
})

In [80]:
report_1 = classification_report(full_df['label'], full_df['predictions_1'], output_dict=True,)
report_2 = classification_report(full_df['label'], full_df['predictions_2'], output_dict=True)
report_3 = classification_report(full_df['label'], full_df['predictions_3'], output_dict=True)

In [82]:
print("Classification Report for Template 1:")
print(report_1)
print("\nClassification Report for Template 2:")
print(report_2)
print("\nClassification Report for Template 3:")
print(report_3)

# Write to .json files
with open('metrics_shape_mistral_template_1.json', 'w') as f:
    pd.DataFrame(report_1).to_json(f, indent=4)
with open('metrics_shape_mistral_template_2.json', 'w') as f:
    pd.DataFrame(report_2).to_json(f, indent=4)
with open('metrics_shape_mistral_template_3.json', 'w') as f:
    pd.DataFrame(report_3).to_json(f, indent=4)

df_1.to_csv('evaluated_shape_mistral_template_1.csv', index=False)
df_2.to_csv('evaluated_shape_mistral_template_2.csv', index=False)
df_3.to_csv('evaluated_shape_mistral_template_3.csv', index=False)

Classification Report for Template 1:
{'0': {'precision': 0.7181208053691275, 'recall': 0.8916666666666667, 'f1-score': 0.7955390334572491, 'support': 120.0}, '1': {'precision': 0.8452380952380952, 'recall': 0.6283185840707964, 'f1-score': 0.7208121827411168, 'support': 113.0}, 'accuracy': 0.7639484978540773, 'macro avg': {'precision': 0.7816794503036113, 'recall': 0.7599926253687316, 'f1-score': 0.7581756080991829, 'support': 233.0}, 'weighted avg': {'precision': 0.7797699631167384, 'recall': 0.7639484978540773, 'f1-score': 0.7592981144404123, 'support': 233.0}}

Classification Report for Template 2:
{'0': {'precision': 0.7272727272727273, 'recall': 0.8666666666666667, 'f1-score': 0.7908745247148289, 'support': 120.0}, '1': {'precision': 0.8222222222222222, 'recall': 0.6548672566371682, 'f1-score': 0.729064039408867, 'support': 113.0}, 'accuracy': 0.7639484978540773, 'macro avg': {'precision': 0.7747474747474747, 'recall': 0.7607669616519175, 'f1-score': 0.759969282061848, 'support': 

In [103]:
import json
import regex as re
def clean_prediction_string(s):
    if pd.isna(s):
        return None

    # Try standard parsing first
    try:
        return json.loads(s)
    except (json.JSONDecodeError, TypeError):
        pass

    # Try to patch common formatting errors
    try:
        # Fix double double-quotes (common from CSV escaping)
        s_fixed = s.replace('""', '"')

        # Fix unquoted keys (e.g. answer: instead of "answer":)
        s_fixed = re.sub(r'([{,])\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*:', r'\1 "\2":', s_fixed)

        # Ensure string values are quoted (very heuristic)
        if '"explanation":' in s_fixed:
            parts = s_fixed.split('"explanation":')
            prefix = parts[0] + '"explanation": '
            rest = ':'.join(parts[1:]).strip()

            # Remove any trailing } or comma outside of quotes
            rest = rest.strip()
            if not rest.startswith('"'):
                rest = '"' + rest
            if not rest.endswith('"'):
                rest = rest.rstrip('}').rstrip(',') + '"'

            s_fixed = prefix + rest + "}"

        return json.loads(s_fixed)

    except Exception as e:
        return None  # Could not fix

In [70]:
df = pd.read_csv('evaluated_shape_mistral_template_3.csv', sep='\t', quotechar='"', doublequote=True, encoding='utf-8')

In [120]:
import regex as re
texts = []
labels = []
outputs = []
for index, row in df.iterrows():
    text = row['text']
    label = row['label']
    output = row['predictions_3']
    output = ast.literal_eval(output)

    if isinstance(output, str):
        output = clean_prediction_string(output)
    texts.append(text)
    labels.append(label)
    outputs.append(output)
new_df = pd.DataFrame({
    'text': texts,
    'label': labels,
    'predictions_3': outputs
})
predictions = [int(outputs.get('answer')) for outputs in outputs]
explanations = [str(output.get('explanation')) for output in outputs]
# new_df.to_csv('evaluated_shape_mistral_template_3.csv', index=False)
# report_3 = classification_report(labels, predictions, output_dict=True)
# print("\nClassification Report for Cleaned Template 3:")
# print(report_3)
# # Write cleaned report to .json file
# with open('metrics_shape_mistral_template_3.json', 'w') as f:
#     pd.DataFrame(report_3).to_json(f, indent=4)

In [123]:
for text, pred, label, explanation in zip(texts, predictions, labels, explanations):
    if pred == 1 and label == 0:
        print(f"Text: {text}\nPredicted: {pred} \nExplanation: {explanation}\nLabel: {label}\n")
        print("-" * 80)

Text: 
Here’s a question, my best friend and I have known each other for about five years now. We are super close and hang out all the time. She vents to me about her boyfriend, who cheats on her and doesn’t take her out much. Whenever I’m around him he’s very nice to me, but whenever it’s just me and her he has told her in the past that it always seems like she likes to hang out with me more than him. During Christmas I was trying to help her with her laundry and he barged into the room trying to help even though I was already helping her. He even recently told her that he doesn’t want me sleeping over their apartment. Am I overthinking this??
Predicted: 1 
Explanation: The post discusses the boyfriend's behavior towards the friend and the friend's mental state regarding the relationship. The boyfriend's actions, such as barging into the room and expressing jealousy, could be causing distress for the friend. The friend is seeking advice on whether she is overthinking the situation, in

In [ ]:


# Example 2
# Text: Just wondering what people are doing to stop breastfeeding? 

# Our daughter is about to turn 1 and is down to only breastfeeding first thing in the morning. From reading bits and pieces, I sort of thought that we would gradually reduce and my supply would reduce and one day I would stop BF her and she would be a Big Girl with her solids and her bits of formula…but it’s not quite working like that! 

# I’ve thought we were done a few times and then ended up in pain with lots of milk and ended up feeding her again. The last time this happened i had to express milk as the baby was staying overnight with her grandparents. I got 150ml of milk just out of my "good boob" after skipping 1 morning of feeding her and have now just continued feeding her each morning. Does this seem normal to people based on your experiences?

# I’ve got a pituitary tumour that has resulted in high prolactin levels in the past so at this point I'm assuming my prolactin isn't dropping enough so I'm going to call endocrinologist on Monday but interested in how “stopping” works for everybody else!
# Predicted: 1 
# Explanation: The post discusses the author's experience with breastfeeding and the challenges they face in weaning their child. They mention their high milk supply and the impact it has on their body, as well as their plans to consult an endocrinologist due to a previous pituitary tumor. These topics are related to mental and physical health, making this a discussion of mental health.
# Label: 0

# Example 3
# Text: Hi everyone. So I went to the doctor today because I have been dealing with stomach issues that are not normal for me. Massive bloating, gas, bowel problems, nausea (and sometimes vomiting) after eating and a lot of other symptoms that have been getting worse since January. 

# Anyways, I went to my doctor today for my check up for my labs for my thyroid and I brought up to her about getting tested for celiac disease. I told her I would like to do the test just to make sure, if it's positive then I know, if its negative then it's ruled out, ya know? I asked for the test and she said sure, but then said I should not eat gluten for a week to see if I start feeling better then we can do the test. She said it would make the results more accurate. But from what I read here and other places online is you should continue eating gluten or else that will not give you an accurate reading.

# My doctor said I can just come in anytime next week to take the blood test (luckily she listens to me when I ask for a test to be done and will order it for me without any issue). But idk, what she said about not eating gluten a week before doesn't sound right.
# Predicted: 1 
# Explanation: The post discusses physical symptoms such as bloating, gas, bowel problems, nausea, and vomiting, which can be indicative of mental health conditions like irritable bowel syndrome (IBS) or other gastrointestinal disorders. Additionally, the individual is seeking medical advice and testing for celiac disease, which is a condition related to digestive health and mental health, as it can lead to anxiety and depression. Therefore, this post can be considered a discussion of mental health, even if the primary focus is on physical symptoms.
# Label: 0

# Example 4
# Text: Just got a response from my email to Sovol about the broken silicone sock and the messed up nozzle.

# It was a pretty timely response and they asked for pics.  I sent the pics today and will await their response.  I'm expecting they'll end at least a new sock and that'll be ok, but a new nozzle or tip or at least some instructions so that it doesn't happen again.

# Just in case anyone didn't catch the prior post, the silicone sock split and fell off.  There was burned filament INSIDE the cover and I emailed Sovol about this.

# IDK if the sock was cracked first and caught some of the filament, or if the nozzle was loose or something is cracked, but clearly not the way it's supposed to work.

# I'll update with their response.  Support is a key thing with any product.
# Predicted: 1 
# Explanation: The post discusses issues with a product (the silicone sock and nozzle for a 3D printer) and the process of contacting the company for a resolution. The mention of the broken sock, burned filament inside the cover, and the need for a response from the company suggests that the individual is experiencing frustration and potentially anxiety or stress related to the issue.
# Label: 0

# Example 5
# Text: 
# Here’s a question, my best friend and I have known each other for about five years now. We are super close and hang out all the time. She vents to me about her boyfriend, who cheats on her and doesn’t take her out much. Whenever I’m around him he’s very nice to me, but whenever it’s just me and her he has told her in the past that it always seems like she likes to hang out with me more than him. During Christmas I was trying to help her with her laundry and he barged into the room trying to help even though I was already helping her. He even recently told her that he doesn’t want me sleeping over their apartment. Am I overthinking this??
# Predicted: 1 
# Explanation: The post discusses the boyfriend's behavior towards the friend and the friend's mental state regarding the relationship. The boyfriend's actions, such as barging into the room and expressing jealousy, could be causing distress for the friend. The friend is seeking advice on whether she is overthinking the situation, indicating she may be struggling with her feelings and the dynamics of the relationship.
# Label: 0

In [124]:
for text, pred, label, explanation in zip(texts, predictions, labels, explanations):
    if pred == 0 and label == 1:
        print(f"Text: {text}\nPredicted: {pred} \nExplanation: {explanation}\nLabel: {label}\n")
        print("-" * 80)

Text: I've constantly been struggling with creating named saves. The saves themselves work fine, and the saves can be named, but the savenames disappear the next time the game boots up.

Example:

\\*Player clicks "Save A"

\\*Save turns into "A: $date\\_and\\_time"

\\*Player clicks load, save displays "A: $date\\_and\\_time"

\\*Player quits game, loads saves

\\*Player goes to load menu, save displays "A: 0"

&amp;#x200B;

What works, but isn't what players commonly do:

\\*Player clicks "Save A"

\\*Save turns into "A: $date\\_and\\_time"

\\*Player returns to game, goes to just one other passage

\\*Player quits game, loads saves

\\*Player goes to load menu, save displays "A: $date\\_and\\_time"

&amp;#x200B;

I've tried a lot of things to try and get around this, but none of them seem to work for all cases. I'm beginning to wonder if I'm overthinking it, because surely it can't be as hard as it's seeming.
Predicted: 0 
Explanation: The post is about a technical issue with saving